In [1]:
print('hi')

hi


In [8]:
from abc import ABC, abstractmethod
from datetime import datetime
import json
import threading
import queue

In [3]:
class LogRecord:
    def __init__(self, level, message, metadata):
        self.level = level
        self.message = message
        self.timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        self.metadata = metadata or {}

In [4]:
class Formatter(ABC):
    @abstractmethod
    def format_message(self, record):
        pass

class SimpleFormatter(Formatter):
    def format_message(self, record):
        metadata_text = self._format_metadata(record.metadata)

        if metadata_text:
            return f"[{record.level}] {record.message} | {metadata_text}"

        return f"[{record.level}] {record.message}"

    def _format_metadata(self, metadata):
        if not metadata:
            return ""

        return " ".join([f"{key}={value}" for key, value in metadata.items()])

class DetailedFormatter(Formatter):
    def format_message(self, record):
        metadata_text = self._format_metadata(record.metadata)

        if metadata_text:
            return f"[{record.timestamp}] [{record.level}] {record.message} | {metadata_text}"

        return f"[{record.timestamp}] [{record.level}] {record.message}"

    def _format_metadata(self, metadata):
        if not metadata:
            return ""

        return " ".join([f"{key}={value}" for key, value in metadata.items()])
        
class JsonFormatter(Formatter):
    def format_message(self, record):
        
        d =  {
            'level' : record.level,
            'message' : record.message,
            'time' : record.timestamp,
            'metadata' : record.metadata
        }

        return json.dumps(d)
        


In [5]:
class Sink(ABC):
    @abstractmethod
    def write(self, message):
        pass

    @abstractmethod
    def get_identity(self):
        pass

    @abstractmethod
    def can_log(self, level, levels):
        pass

In [7]:
class ConsoleSink(Sink):
    def __init__(self, minimum_level):
        self.lock = threading.Lock()
        self.minimum_level = minimum_level
        
    def write(self, message):
        with self.lock:
            print(f"Console {message}")

    def get_identity(self):
        return "console"
    
    def can_log(self, level, levels):
        return levels[level] >= levels[self.minimum_level]
        

class FileSink(Sink):
    def __init__(self, file_path : str, minimum_level):
        self.lock = threading.Lock()
        self.minimum_level = minimum_level
        self.file_path = file_path
        
    def write(self,  message):
        with self.lock:
            with open(self.file_path, "a", encoding="utf-8") as file:
                file.write(message + "\n")

    def get_identity(self):
        return f"file:{self.file_path}"
    
    def can_log(self, level, levels):
        return levels[level] >= levels[self.minimum_level]

class DatabaseSink(Sink):
    def __init__(self, database, table, minimum_level):
        self.minimum_level = minimum_level
        self.database = database
        self.table = table

    def write(self, message):
        print(f"Inserting into {self.database}.{self.table}: {message}")

    def get_identity(self):
        return f"database:{self.database}:{self.table}"
    
    def can_log(self, level, levels):
        return levels[level] >= levels[self.minimum_level]



In [11]:
class Logger:
    __instance = None
    LEVELS = {
    "DEBUG": 1,
    "INFO": 2,
    "WARNING": 3,
    "ERROR": 4
    }
    __lock = threading.Lock()

    def __new__(cls):
        if cls.__instance is None:
            with cls.__lock:
                if cls.__instance is None:
                    cls.__instance = super(Logger, cls).__new__(cls)
        return cls.__instance

    def __init__(self):
        if hasattr(self, "_initialized"):
            return

        self.__minimum_level = "DEBUG"
        self.sinks = []
        self.sinks_lock = threading.Lock()

        self.formatter = SimpleFormatter()
        self.add_sink(ConsoleSink("DEBUG"))

        self.log_queue = queue.Queue()
        self.is_running = True

        self.worker_thread = threading.Thread(target=self._process_logs, daemon=True)
        self.worker_thread.start()

        print("Logger initialized")
        self._initialized = True


    @classmethod
    def get_instance(cls):
        return cls()
    
    def set_minimum_level(self, level):
        if level in self.LEVELS:
            self.__minimum_level = level
        else:
            raise ValueError("Invalid log level")
        

    def add_sink(self, sink: Sink):
        if not isinstance(sink, Sink):
            raise TypeError("sink must be an instance of Sink")
        
        with self.sinks_lock:
            for existing_sink in self.sinks:
                if existing_sink.get_identity() == sink.get_identity():
                    print("Sink already added")
                    return

            self.sinks.append(sink)

    def remove_sink(self, sink : Sink):
        if not isinstance(sink, Sink):
            raise TypeError("sink must be an instance of Sink")
        
        with self.sinks_lock:
            for existing_sink in self.sinks:
                if existing_sink.get_identity() == sink.get_identity():
                    self.sinks.remove(existing_sink)
                    print("removed")
                    return
                
            print("doesn't exist")

    def clear_sinks(self):
        with self.sinks_lock:
            self.sinks.clear()

        
        
        

    def set_formatter(self, formatter: Formatter):
        if not isinstance(formatter, Formatter):
            raise TypeError("formatter must be an instance of Formatter")

        self.formatter = formatter

    def log(self, level, message, metadata=None):
        if self.LEVELS.get(level, 0) >= self.LEVELS.get(self.__minimum_level, 0):
            record = LogRecord(level, message, metadata)
            self.log_queue.put(record)

    def _process_logs(self):
        while self.is_running or not self.log_queue.empty():
            try:
                record = self.log_queue.get(timeout=0.5)

                formatted_message = self.formatter.format_message(record)

                with self.sinks_lock:
                    sinks_snapshot = self.sinks[:]

                for sink in sinks_snapshot:
                    if sink.can_log(record.level, self.LEVELS):
                        sink.write(formatted_message)

                self.log_queue.task_done()

            except queue.Empty:
                continue

    def flush(self):
        self.log_queue.join()

    def shutdown(self):
        self.is_running = False
        self.log_queue.join()
        self.worker_thread.join()


    def info(self, message, metadata = None):
        self.log("INFO", message, metadata)

    def error(self, message, metadata = None):
        self.log("ERROR", message, metadata)

    def debug(self, message , metadata = None):
        self.log("DEBUG", message, metadata)

    def warning(self, message, metadata = None):
        self.log("WARNING", message, metadata)

In [12]:
import os
import threading


# -----------------------------
# Helper functions
# -----------------------------

def delete_file_if_exists(file_path):
    if os.path.exists(file_path):
        os.remove(file_path)


def count_lines(file_path):
    if not os.path.exists(file_path):
        return 0

    with open(file_path, "r", encoding="utf-8") as file:
        return len(file.readlines())


def print_file_content(file_path):
    print(f"\n--- {file_path} content ---")

    if not os.path.exists(file_path):
        print("File does not exist")
        return

    with open(file_path, "r", encoding="utf-8") as file:
        print(file.read())


# -----------------------------
# Clean old files
# -----------------------------

for file_name in [
    "app.log",
    "error.log",
    "audit.log",
    "async_app.log",
    "async_error.log"
]:
    delete_file_if_exists(file_name)


# -----------------------------
# TEST 1: Singleton
# -----------------------------

print("\n========== TEST 1: Singleton ==========")

logger1 = Logger.get_instance()
logger2 = Logger.get_instance()

print("logger1 is logger2:", logger1 is logger2)  # Expected: True

logger = logger1


# -----------------------------
# TEST 2: Basic setup
# -----------------------------

print("\n========== TEST 2: Basic Setup ==========")

logger.flush()
logger.clear_sinks()

logger.set_minimum_level("DEBUG")
logger.set_formatter(DetailedFormatter())

logger.add_sink(ConsoleSink("DEBUG"))
logger.add_sink(FileSink("app.log", "INFO"))
logger.add_sink(FileSink("error.log", "ERROR"))

logger.debug("Debug log should go only to console", metadata={"step": 1})
logger.info("User logged in", metadata={"user_id": 101})
logger.warning("API is slow", metadata={"latency_ms": 1200})
logger.error("Payment failed", metadata={"user_id": 101, "order_id": 999})

logger.flush()

print("app.log lines after TEST 2:", count_lines("app.log"))        # Expected: 3
print("error.log lines after TEST 2:", count_lines("error.log"))    # Expected: 1


# -----------------------------
# TEST 3: Duplicate sink
# -----------------------------

print("\n========== TEST 3: Duplicate Sink ==========")

logger.add_sink(FileSink("app.log", "INFO"))       # Expected: Sink already added
logger.add_sink(FileSink("error.log", "ERROR"))    # Expected: Sink already added

logger.flush()


# -----------------------------
# TEST 4: Multiple file sinks
# -----------------------------

print("\n========== TEST 4: Multiple File Sinks ==========")

logger.add_sink(FileSink("audit.log", "WARNING"))

logger.info("This should go to console and app.log only")
logger.warning("This should go to console, app.log, and audit.log")
logger.error("This should go to console, app.log, error.log, and audit.log")

logger.flush()

print("app.log lines after TEST 4:", count_lines("app.log"))        # Expected: 6
print("error.log lines after TEST 4:", count_lines("error.log"))    # Expected: 2
print("audit.log lines after TEST 4:", count_lines("audit.log"))    # Expected: 2


# -----------------------------
# TEST 5: Remove sink
# -----------------------------

print("\n========== TEST 5: Remove Sink ==========")

logger.remove_sink(FileSink("error.log", "ERROR"))

logger.error(
    "This error should NOT go to error.log after remove",
    metadata={"test": "remove_sink"}
)

logger.flush()

print("app.log lines after TEST 5:", count_lines("app.log"))        # Expected: 7
print("error.log lines after TEST 5:", count_lines("error.log"))    # Expected: 2
print("audit.log lines after TEST 5:", count_lines("audit.log"))    # Expected: 3


# -----------------------------
# TEST 6: JSON formatter
# -----------------------------

print("\n========== TEST 6: JSON Formatter ==========")

logger.flush()
logger.set_formatter(JsonFormatter())

logger.info("JSON info log", metadata={"user_id": 202})
logger.error("JSON error log", metadata={"order_id": 555})

logger.flush()

print("app.log lines after TEST 6:", count_lines("app.log"))        # Expected: 9
print("audit.log lines after TEST 6:", count_lines("audit.log"))    # Expected: 4


# -----------------------------
# TEST 7: Clear sinks
# -----------------------------

print("\n========== TEST 7: Clear Sinks ==========")

logger.flush()
logger.clear_sinks()

logger.error("This should not print or write anywhere")

logger.flush()

print("If you see no log above this line, clear_sinks works.")


# -----------------------------
# TEST 8: Async + Threading
# -----------------------------

print("\n========== TEST 8: Async Threading Test ==========")

logger.flush()
logger.clear_sinks()

logger.set_formatter(DetailedFormatter())
logger.set_minimum_level("DEBUG")

logger.add_sink(ConsoleSink("ERROR"))
logger.add_sink(FileSink("async_app.log", "DEBUG"))
logger.add_sink(FileSink("async_error.log", "ERROR"))


def worker(thread_id):
    for i in range(5):
        logger.info(
            f"Info log from thread {thread_id}",
            metadata={"thread_id": thread_id, "count": i}
        )

        logger.error(
            f"Error log from thread {thread_id}",
            metadata={"thread_id": thread_id, "count": i}
        )


threads = []

for t in range(5):
    thread = threading.Thread(target=worker, args=(t,))
    threads.append(thread)
    thread.start()

for thread in threads:
    thread.join()

logger.flush()

print("Async logging test completed")

print("async_app.log lines:", count_lines("async_app.log"))          # Expected: 50
print("async_error.log lines:", count_lines("async_error.log"))      # Expected: 25


# -----------------------------
# TEST 9: Final file verification
# -----------------------------

print("\n========== TEST 9: Final File Count Verification ==========")

print("app.log lines:", count_lines("app.log"))                  # Expected: 9
print("error.log lines:", count_lines("error.log"))              # Expected: 2
print("audit.log lines:", count_lines("audit.log"))              # Expected: 4
print("async_app.log lines:", count_lines("async_app.log"))      # Expected: 50
print("async_error.log lines:", count_lines("async_error.log"))  # Expected: 25


# Optional: view file contents
# print_file_content("app.log")
# print_file_content("error.log")
# print_file_content("audit.log")
# print_file_content("async_app.log")
# print_file_content("async_error.log")


# -----------------------------
# Shutdown at the very end
# -----------------------------

logger.shutdown()

print("\n========== ALL TESTS COMPLETED ==========")


========== TEST 1: Singleton ==========
Logger initialized
logger1 is logger2: True

========== TEST 2: Basic Setup ==========
Console [2026-06-18 20:20:03] [DEBUG] Debug log should go only to console | step=1
Console [2026-06-18 20:20:03] [INFO] User logged in | user_id=101
Console [2026-06-18 20:20:03] [WARNING] API is slow | latency_ms=1200
Console [2026-06-18 20:20:03] [ERROR] Payment failed | user_id=101 order_id=999
app.log lines after TEST 2: 3
error.log lines after TEST 2: 1

========== TEST 3: Duplicate Sink ==========
Sink already added
Sink already added

========== TEST 4: Multiple File Sinks ==========
Console [2026-06-18 20:20:03] [INFO] This should go to console and app.log only
Console [2026-06-18 20:20:03] [WARNING] This should go to console, app.log, and audit.log
Console [2026-06-18 20:20:03] [ERROR] This should go to console, app.log, error.log, and audit.log
app.log lines after TEST 4: 6
error.log lines after TEST 4: 2
audit.log lines after TEST 4: 2

========== T